In [ ]:
from transformers import CLIPConfig, CLIPModel

# Load the configuration or the model
model_id = "openai/clip-vit-base-patch32"
config = CLIPConfig.from_pretrained(model_id)

# 1. Shared multimodal projection dimension (often what is meant by 'CLIP embedding')
print(f"Shared Projection Dimension: {config.projection_dim}")

# 2. Internal vision encoder hidden dimension
print(f"Vision Hidden Dimension: {config.vision_config.hidden_size}")

# 3. Internal text encoder hidden dimension
print(f"Text Hidden Dimension: {config.text_config.hidden_size}")

In [ ]:
import torch

def inspect_checkpoint(filepath='./outputs/clip_qwen_0.8B_clip_base/best_metric_model.pt'):
    print(f"Loading checkpoint from: {filepath}\n")
    state_dict = torch.load(filepath, map_location='cpu')
    
    keys = list(state_dict.keys())
    print(f"Total number of tensors in file: {len(keys)}")
    
    # Save the full output to a text file so you can easily Ctrl+F through it
    out_file = "checkpoint_keys.txt"
    with open(out_file, "w") as f:
        for k in keys:
            shape_str = str(list(state_dict[k].shape))
            f.write(f"{k}  ---  {shape_str}\n")
            
    print(f"Full list of keys and shapes saved to: {out_file}\n")
    
    print("--- FIRST 20 KEYS ---")
    for k in keys[:20]:
        print(k)
        
    print("\n--- LAST 20 KEYS ---")
    for k in keys[-20:]:
        print(k)
        
    # Specifically look for what might be causing those 48 unexpected keys
    encoder_keys = [k for k in keys if k.startswith('encoder.')]
    print(f"\nTotal keys starting with 'encoder.': {len(encoder_keys)}")

if __name__ == "__main__":
    inspect_checkpoint()

In [7]:
from transformers import AutoConfig

model_name = "Qwen/Qwen3.5-9B-Base"

print(f"Fetching configuration for {model_name}...")
config = AutoConfig.from_pretrained(model_name)

# Accessing parameters from the nested text_config
text_cfg = config.text_config

print("-" * 30)
print(f"Hidden Size (Embedding Dim): {text_cfg.hidden_size}")
print(f"Number of Hidden Layers:     {text_cfg.num_hidden_layers}")
print(f"Intermediate Size (MLP):    {text_cfg.intermediate_size}")
print(f"Vocabulary Size:            {text_cfg.vocab_size}")
print("-" * 30)

# Check if it matches your projection layer logic
input_dim = 768 # Your CLIP output
print(f"Your embed_proj will be: nn.Linear({input_dim}, {text_cfg.hidden_size})")

Fetching configuration for Qwen/Qwen3.5-9B-Base...
------------------------------
Hidden Size (Embedding Dim): 4096
Number of Hidden Layers:     32
Intermediate Size (MLP):    12288
Vocabulary Size:            248320
------------------------------
Your embed_proj will be: nn.Linear(768, 4096)


In [11]:
import evaluate

def analyze_hf_meteor(references, prediction, model_name):
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    print("References (GT):")
    for i, ref in enumerate(references):
        print(f"  {i+1}. {ref}")
    print(f"\nPrediction:       {prediction}")
    
    # Load the exact metric used in train.py
    meteor = evaluate.load('meteor')
    
    # Calculate overall score against all references (METEOR takes the max internally)
    results = meteor.compute(predictions=[prediction], references=[references])
    
    # --- Find the closest GT by evaluating individually ---
    best_score = -1
    best_ref = ""
    
    print("\n--- Individual Reference Scores ---")
    for i, ref in enumerate(references):
        # evaluate expects references as a list of lists: [[ref]]
        indiv_score = meteor.compute(predictions=[prediction], references=[[ref]])['meteor']
        print(f"  GT {i+1} Score: {indiv_score*100:5.2f}% -> {ref}")
        
        if indiv_score > best_score:
            best_score = indiv_score
            best_ref = ref
            
    print(f"\n>>> CLOSEST GT MATCH: '{best_ref}' <<<")
    print(f">>> MATCH SCORE:      {best_score * 100:.2f}% <<<")
    
    # --- Manual Mathematical Demonstration ---
    pred_words = len(prediction.split())
    ref_words = [len(ref.split()) for ref in references]
    avg_ref_words = sum(ref_words) / len(ref_words)
    
    print("\n--- Diagnostic Breakdown ---")
    print(f"Average GT Length: {avg_ref_words:.1f} words")
    print(f"Prediction Length: {pred_words} words")
    
    ratio = pred_words / avg_ref_words
    print(f"Verbosity Ratio:   {ratio:.2f}x the size of the average ground truth!")
    
    if ratio > 2.0:
        print(">> WARNING: Prediction is massively verbose. METEOR Precision will be crushed!")
    elif ratio < 1.0:
        print(">> NOTE: Prediction is very short. Safe from verbosity penalties.")
        
    print(f"\n>>> FINAL AGGREGATE METEOR SCORE: {results['meteor'] * 100:.2f}% <<<")

if __name__ == "__main__":
    
    vizwiz_gt = [
        "a hand on top of a can of salmon that is on a marble kitchen counter",
            "A tin can of Sno-Tip brand Wild Alaska chum salmon with someone's hand on top of it.",
            "A can of Snow tip wild Alaska chum salmon.",
            "A person has a can of fish on the counter.",
            "Aluminum can with salmon placed on granite countertop."
    ]
    
    # Scenario 1: The 0.8B Fine-Tuned Model
    pred_0_8b = "A box of frozen food is on a counter."
    analyze_hf_meteor(vizwiz_gt, pred_0_8b, "0.8B Fine-Tuned")
    
    # Scenario 2: The 9B Zero-Shot Model
    pred_9b = "A can of Campbell’s cream of mushroom soup."
    analyze_hf_meteor(vizwiz_gt, pred_9b, "9B Zero-Shot")


Model: 0.8B Fine-Tuned
References (GT):
  1. a hand on top of a can of salmon that is on a marble kitchen counter
  2. A tin can of Sno-Tip brand Wild Alaska chum salmon with someone's hand on top of it.
  3. A can of Snow tip wild Alaska chum salmon.
  4. A person has a can of fish on the counter.
  5. Aluminum can with salmon placed on granite countertop.

Prediction:       A box of frozen food is on a counter.


[nltk_data] Downloading package wordnet to /home/pau/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/pau/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/pau/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



--- Individual Reference Scores ---
  GT 1 Score: 33.19% -> a hand on top of a can of salmon that is on a marble kitchen counter
  GT 2 Score: 11.05% -> A tin can of Sno-Tip brand Wild Alaska chum salmon with someone's hand on top of it.
  GT 3 Score: 15.00% -> A can of Snow tip wild Alaska chum salmon.
  GT 4 Score: 39.12% -> A person has a can of fish on the counter.
  GT 5 Score: 10.99% -> Aluminum can with salmon placed on granite countertop.

>>> CLOSEST GT MATCH: 'A person has a can of fish on the counter.' <<<
>>> MATCH SCORE:      39.12% <<<

--- Diagnostic Breakdown ---
Average GT Length: 12.0 words
Prediction Length: 9 words
Verbosity Ratio:   0.75x the size of the average ground truth!
>> NOTE: Prediction is very short. Safe from verbosity penalties.

>>> FINAL AGGREGATE METEOR SCORE: 39.12% <<<

Model: 9B Zero-Shot
References (GT):
  1. a hand on top of a can of salmon that is on a marble kitchen counter
  2. A tin can of Sno-Tip brand Wild Alaska chum salmon with someone'

[nltk_data] Downloading package wordnet to /home/pau/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/pau/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/pau/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import nltk
import ssl

# Bypass SSL certificate verification (very common issue in server environments)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download the specific packages
print("Downloading NLTK packages...")
nltk.download('punkt')
nltk.download('punkt_tab')  # Required for newer NLTK versions
nltk.download('wordnet')
nltk.download('omw-1.4')
print("Downloads complete!")